In [ ]:
# NBS-Predictor: Post-Processing
### Extract NBS values from each model for each lake and month, calculate probabilities of exceedence (POEs)
##### Developed on 8/26/2025 by J.L. Ward (USACE-Detroit)
##### Last modified on 8/19/2026

In [ ]:
""" Purpose: 
Extract NBS values (units: cms) from each of the 3 statistical models (Random Forest (RF), Gaussian Processes (GP), and XG Boost (XGB).
Put them in a format compatible with the forecasting framework such that they can be used in PickSupplies.txt
"""

' Purpose: \nExtract NBS values (units: cms) from each of the 4 statistical models (Random Forest (RF), Neural Network (NN), Gaussian Processes (GP), and XG Boost (XGB).\nPut them in a format compatible with the forecasting framework such that they can be used in PickSupplies.txt\n'

In [3]:
# Import Modules
import numpy as np
import pandas as pd
# import sqlite3 # for opening database files.           # Jamie note (7/31/2026): for now, getting rid of database functionality. 

In [ ]:
# User inputs: Bulletin Month and Year
bulletin_year = '2026' # User: enter the year of the bulletin.
bulletin_month = 'AUG' # User: enter the month of the bulletin. 
num_months_to_forecast = 6 # Can be an integer between 1 and 9.
cms_or_mm = 'cms' # Output in mm or cms.
write_flag = 1 # 0 (don't write), 1 (do write)

# DON'T CHANGE (Jamie: 7/31/2026)
database_or_csv = 'csv' # Options: 'database' or 'csv'
# End don't change

# END User Inputs

# Database, csv input directories; filenames 
in_share_dir = ''   # Keep single quotes. Where NBS-predictor database is located.
archive_dir = ''    # Delete single quotes.

# Path to input CFS forecast database
# cnbs_database = in_share_dir + 'cnbs_forecast.db'
cnbs_forecast = archive_dir + 'cnbs_forecast.csv'

# Path to output POE and NBS databases (containing 5%, 50%, and 95% info for each lake)
if cms_or_mm == 'mm':
    cnbs_components_processed_ends = ['_cnbs_5_50_95_mm.csv', '_prec_5_50_95_mm.csv', '_evap_5_50_95_mm.csv', '_runo_5_50_95_mm.csv'] # CNBS, precipitation, evapoation, runoff
else:
    cnbs_components_processed_ends = ['_cnbs_5_50_95.csv', '_prec_5_50_95.csv', '_evap_5_50_95.csv', '_runo_5_50_95.csv']

In [5]:
# Function: open cnbs_forecast.db
# Jamie note (7/31/2026): not using right now, but could in the future (with some tweaks).
def open_db(cfs_database):
    conn = sqlite3.connect(cfs_database)

    # Define the query to get all the data
    query = '''
    SELECT * FROM cfs_forecast_data
    '''
    
    # Execute the query and fetch the data into a DataFrame
    df = pd.read_sql(query, conn)
    
    # Close the connection once done
    conn.close()
    
    df.set_index(['cfs_run', 'month', 'year'], drop=True, inplace=True)

    return df

# Function: calculate median, 5%, and 95%. upper_bound will be assigned to the 5% column (for probability of exceedence)
# Code is courtesy of Lindsay Fitzpatrick (CIGLR). 
def calculate_median_confidence(df,lake):
    median = df.loc[:,lake].median() # Calculate the median of the dataframe in question.

    # Calculate the 2.5th and 97.5th percentiles for the confidence band
    lower_bound = df.loc[:,lake].quantile(0.05)
    upper_bound = df.loc[:,lake].quantile(0.95) 

    return median, lower_bound, upper_bound # Outputs (in order): median (50%), lower_bound (95%), and upper_bound(5%)


In [ ]:
# Read in data
if database_or_csv == 'csv': # reading in csv data.
    print('Reading in cnbs_forecast.csv')
    df = pd.read_csv(cnbs_forecast)
else: # reading in database data. Use open_db function.
    print('Reading in cnbs_forecast.db')
    #df = open_db(cnbs_database)
    conn = sqlite3.connect(cnbs_database)
    # Define the query to get all the data
    query = '''
    SELECT * FROM cnbs_forecast
    '''
    
    # Execute the query and fetch the data into a DataFrame
    df = pd.read_sql(query, conn)

    # Close the connection once done
    conn.close()
    
    df.set_index(['cfs_run', 'month', 'year'], drop=True, inplace=True)

# print(df)

Reading in cnbs_forecast.csv
           cfs_run  month  year model            lake      component  \
0       2025100100      7  2026    GP            erie  precipitation   
1       2025100100      7  2026    GP  michigan-huron  precipitation   
2       2025100100      7  2026    GP         ontario  precipitation   
3       2025100100      7  2026    GP        superior  precipitation   
4       2025100100      8  2026    GP            erie  precipitation   
...            ...    ...   ...   ...             ...            ...   
535995  2026072018      5  2027   XGB        superior            nbs   
535996  2026072018      6  2027   XGB            erie            nbs   
535997  2026072018      6  2027   XGB  michigan-huron            nbs   
535998  2026072018      6  2027   XGB         ontario            nbs   
535999  2026072018      6  2027   XGB        superior            nbs   

        value [mm]  value [cms]  
0        80.086448   767.106416  
1        75.706245  3315.907518  
2   

In [ ]:
# Pre-defined dictionaries and lists containing month, POE threshold info
comp_varnames = ['nbs', 'precipitation', 'evaporation', 'runoff'] # Component variable names; doens't change

models = ['GP', 'RF', 'XGB']                        # In order, Gaussian Processes, Random Forest, and XG Boost.
lake_indices = ['sup', 'mih', 'eri', 'ont']         # Used to extract lake information from dictionaries and dataframes.
thresh_names = ['5%', '50%', '95%']                 # POEs to be listed in the final dataframes.

# Create multiIndex (for columns) using lake_indices and thresh_names
column_multiIndex = pd.MultiIndex.from_product([lake_indices, thresh_names], names=['Lake', 'Threshold'])

# Define lake names as they appear in cnbs_forecast.csv/db
in_lake_names = {lake_indices[0]:'superior', lake_indices[1]:'michigan-huron', lake_indices[2]:'erie', lake_indices[3]:'ontario'} # csv and db lake identifiers.

# month names and their corresponding numbers.
start_month_dict = {'DEC': 12, 'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6, 'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11} 

# Now that start_month_dict has been defined, figure out which months forecast months will be considered, based on the values of bulletin_month and num_months_to_forecast
months = [start_month_dict[bulletin_month]] # create list (months) with the first entry being the first month of the forecast.
for index in range(1, num_months_to_forecast):
    new_month = (months[index-1] + 1) % 12 # the next month will be the previous month number, plus 1. If the previous month is 12, the next month is 1. Therefore, use modulus 12 operator.
    if new_month == 0: # December, 12 % 12 == 0.
        new_month = 12 # Manually set December.

    # Append new_month to the end of months
    months.append(new_month)

In [8]:
# Calculate 5, 50, and 95% CNBS for each lake, for each month. 
for model in models:
    forecast_nbs_dict = {}
    for comp_var in comp_varnames:
        forecast_nbs_dict[comp_var] = pd.DataFrame(data=None, index=months, columns=column_multiIndex) # define new dataframe to hold NBS values.
        for lake in lake_indices:
            for month in months:
                if cms_or_mm == 'cms':
                    # Calculate 50%, 95%, and 5% NBS values. This is done separately for each model (GP, LR, NN, and RF), month, and lake 
                    forecast_nbs_dict[comp_var].loc[month, (lake,thresh_names[1])], forecast_nbs_dict[comp_var].loc[month, (lake,thresh_names[2])], forecast_nbs_dict[comp_var].loc[month, (lake,thresh_names[0])] = \
                    calculate_median_confidence(df.loc[((df['month']==month) & (df['model']==model) & (df['component']==comp_var) & (df['lake']==in_lake_names[lake])),:], \
                                                                               'value [cms]')
                else: # mm
                    # Calculate 50%, 95%, and 5% NBS values. This is done separately for each model (GP, LR, NN, and RF), month, and lake 
                    forecast_nbs_dict[comp_var].loc[month, (lake,thresh_names[1])], forecast_nbs_dict[comp_var].loc[month, (lake,thresh_names[2])], forecast_nbs_dict[comp_var].loc[month, (lake,thresh_names[0])] = \
                    calculate_median_confidence(df.loc[((df['month']==month) & (df['model']==model) & (df['component']==comp_var) & (df['lake']==in_lake_names[lake])),:], \
                                                                               'value [mm]')

    # Once NBS/Precip/Evap/Runoff has been calculated, write to csv (if write_flag = 1)
    if write_flag == 1:
        for nbs_index in [0, 1, 2, 3]:
            forecast_nbs_dict[comp_varnames[nbs_index]].to_csv(archive_dir + model + '_' + str(num_months_to_forecast) + 'mos' + cnbs_components_processed_ends[nbs_index])